#### Method 1: Complex Form (LLaMa)

In [1]:
import torch

In [5]:
def precompute_freqs_cis(dim:int, seq_len:int, theta:float = 10000.0):
    """
    预计算 RoPE 的复数频率。

    Args:
        dim: 每个注意力头的维度 d_head
        seq_len: 最大序列长度
        theta: 频率基数（默认 10000）

    Returns:
        freqs_cis: (seq_len, dim//2) 的复数张量
    """
    # step 1:计算频率 θ_i = 1 / (10000^(2i/d))
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2) / dim)) #shape: [dim/2]
    # Step 2: 计算外积 m * θ_i，得到每个位置每个维度的旋转角度
    t = torch.arange(seq_len)
    freqs = torch.outer(t, freqs) #(seq_len, dim/2)
    # Step 3: 转为复数 e^{i * m * θ_i}
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


In [ ]:
def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor
) -> tuple:
    """
    将 RoPE 应用到 Query 和 Key 上。

    Args:
        xq: (batch, seq_len, n_heads, head_dim) — Query
        xk: (batch, seq_len, n_kv_heads, head_dim) — Key
        freqs_cis: (seq_len, head_dim//2) — 预计算的复数频率

    Returns:
        xq_out, xk_out: 旋转后的 Q, K，形状不变
    """
    # Step 1: 将实数张量视为复数
    # (B, S, H, D) → (B, S, H, D//2)
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xq.shape[:-1], -1, 2))

    # Step 2: 调整 freqs_cis 形状以广播
    # (T, D//2) → (1, T, 1, D//2)
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)

    # Step 3: 复数乘法 = 旋转
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(-2)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(-2)

    return xq_out.type_as(xq), xk_out.type_as(xk)

#### xq: (B, T, 32, 128)  → reshape → (B, T, 32, 64, 2)  → view_as_complex → (B, T, 32, 64)
#### freqs_cis: (T, 64)    → unsqueeze → (1, T, 1, 64)
#### xq_ * freqs_cis:      → (B, T, 32, 64) [complex]
#### → view_as_real:        → (B, T, 32, 64, 2)
#### → flatten(-2):         → (B, T, 32, 128)

#### Method 2: Rotation matrix form (HuggingFace)

In [ ]:
def rotary_emb_huggingface(x, cos, sin):
    """
    HuggingFace Transformers 中的 RoPE 实现方式。

    x:   (batch, n_heads, seq_len, head_dim)
    cos: (seq_len, head_dim)
    sin: (seq_len, head_dim)
    """
    def rotate_half(x):
        """将 x 的后半部分取反并与前半部分交换。"""
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    return (x * cos) + (rotate_half(x) * sin)